In [1]:
%pip install pandas requests python-dotenv matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 33.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 19.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 32.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [matplotlib]7 [matplotlib]

[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))

CENSUS_API_KEY = os.getenv("CENSUS_API_KEY")

if not CENSUS_API_KEY:
    raise ValueError("CENSUS_API_KEY is missing from .env")

Matplotlib is building the font cache; this may take a moment.


In [3]:
url = "https://api.census.gov/data/2024/acs/acs5"

params = {
    "get": (
        "NAME,"
        "B19013_001E,"
        "B08201_001E,"
        "B08201_002E,"
        "B08301_001E,"
        "B08301_010E"
    ),
    "for": "tract:*",
    "in": "state:06 county:075",
    "key": CENSUS_API_KEY
}

response = requests.get(url, params=params, timeout=60)
response.raise_for_status()

data = response.json()

acs = pd.DataFrame(data[1:], columns=data[0])

acs.head(10)

,NAME,B19013_001E,B08201_001E,B08201_002E,B08301_001E,B08301_010E,state,county,tract
0,Census Tract 101.01; San Francisco County; Cal...,89022,1216,451,1167,216,06,075,010101
1,Census Tract 101.02; San Francisco County; Cal...,102826,930,347,931,133,06,075,010102
2,Census Tract 102.01; San Francisco County; Cal...,156017,1399,454,1812,268,06,075,010201
3,Census Tract 102.02; San Francisco County; Cal...,176958,1099,150,1284,119,06,075,010202
4,Census Tract 103; San Francisco County; Califo...,158144,2060,455,2655,188,06,075,010300
5,Census Tract 104.01; San Francisco County; Cal...,181705,953,355,1299,108,06,075,010401
6,Census Tract 104.02; San Francisco County; Cal...,203354,1233,329,1280,68,06,075,010402
7,Census Tract 105; San Francisco County; Califo...,179896,1848,636,1869,300,06,075,010500
8,Census Tract 106; San Francisco County; Califo...,75938,1723,1217,1779,616,06,075,010600
9,Census Tract 107.01; San Francisco County; Cal...,24854,1792,1531,957,447,06,075,010701


In [4]:
# RENAME COLS
acs = acs.rename(columns={
    "NAME": "tract_name",
    "B19013_001E": "median_household_income",
    "B08201_001E": "total_households",
    "B08201_002E": "households_without_vehicle",
    "B08301_001E": "total_workers",
    "B08301_010E": "public_transit_commuters"
})

In [10]:
acs.head(10)


,tract_name,median_household_income,total_households,households_without_vehicle,total_workers,public_transit_commuters,state,county,tract
0,Census Tract 101.01; San Francisco County; Cal...,89022,1216,451,1167,216,06,075,010101
1,Census Tract 101.02; San Francisco County; Cal...,102826,930,347,931,133,06,075,010102
2,Census Tract 102.01; San Francisco County; Cal...,156017,1399,454,1812,268,06,075,010201
3,Census Tract 102.02; San Francisco County; Cal...,176958,1099,150,1284,119,06,075,010202
4,Census Tract 103; San Francisco County; Califo...,158144,2060,455,2655,188,06,075,010300
5,Census Tract 104.01; San Francisco County; Cal...,181705,953,355,1299,108,06,075,010401
6,Census Tract 104.02; San Francisco County; Cal...,203354,1233,329,1280,68,06,075,010402
7,Census Tract 105; San Francisco County; Califo...,179896,1848,636,1869,300,06,075,010500
8,Census Tract 106; San Francisco County; Califo...,75938,1723,1217,1779,616,06,075,010600
9,Census Tract 107.01; San Francisco County; Cal...,24854,1792,1531,957,447,06,075,010701


In [12]:
# CONVERT TO PROPER DTYPES and CREATE PERCENTAGE COLS
numeric_cols = [
    "median_household_income",
    "total_households",
    "households_without_vehicle",
    "total_workers",
    "public_transit_commuters"
]

acs[numeric_cols] = acs[numeric_cols].apply(
    pd.to_numeric, errors="coerce"
)

In [13]:
acs["pct_without_vehicle"] = (
    acs["households_without_vehicle"]
    / acs["total_households"].replace(0, np.nan) * 100
)

acs["pct_transit_commuters"] = (
    acs["public_transit_commuters"]
    / acs["total_workers"].replace(0, np.nan) * 100
)

In [14]:
acs.head(10)

,tract_name,median_household_income,total_households,households_without_vehicle,total_workers,public_transit_commuters,state,county,tract,pct_without_vehicle,pct_transit_commuters
0,Census Tract 101.01; San Francisco County; Cal...,89022,1216,451,1167,216,06,075,010101,37.088816,18.508997
1,Census Tract 101.02; San Francisco County; Cal...,102826,930,347,931,133,06,075,010102,37.311828,14.285714
2,Census Tract 102.01; San Francisco County; Cal...,156017,1399,454,1812,268,06,075,010201,32.451751,14.790287
3,Census Tract 102.02; San Francisco County; Cal...,176958,1099,150,1284,119,06,075,010202,13.648772,9.267913
4,Census Tract 103; San Francisco County; Califo...,158144,2060,455,2655,188,06,075,010300,22.087379,7.080979
5,Census Tract 104.01; San Francisco County; Cal...,181705,953,355,1299,108,06,075,010401,37.250787,8.314088
6,Census Tract 104.02; San Francisco County; Cal...,203354,1233,329,1280,68,06,075,010402,26.682887,5.312500
7,Census Tract 105; San Francisco County; Califo...,179896,1848,636,1869,300,06,075,010500,34.415584,16.051364
8,Census Tract 106; San Francisco County; Califo...,75938,1723,1217,1779,616,06,075,010600,70.632618,34.626194
9,Census Tract 107.01; San Francisco County; Cal...,24854,1792,1531,957,447,06,075,010701,85.435268,46.708464


In [16]:
acs.shape

(244, 11)